In [1]:
!git clone https://github.com/Shiba-dotcom/waste-detection2-Stage.git

Cloning into 'waste-detection2-Stage'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 43 (delta 6), reused 43 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 8.61 MiB | 6.98 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [2]:
!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/TrashNet
!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/RealWaste
!mkdir -p /kaggle/working/waste-detection2-Stage/data/raw

In [3]:
import os
import shutil

# Cấu hình danh sách các cặp đường dẫn (Nguồn -> Đích)
datasets_to_copy = [
    {
        "src": "/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized",
        "dst": "/kaggle/working/waste-detection2-Stage/data/external/TrashNet"
    },
    {
        "src": "/kaggle/input/datasets/sohamchaudhari2004/taco-trash-detection-dataset",
        "dst": "/kaggle/working/waste-detection2-Stage/data/raw"
    },
    {
        "src": "/kaggle/input/datasets/joebeachcapital/realwaste/realwaste-main/RealWaste",
        "dst": "/kaggle/working/waste-detection2-Stage/data/external/RealWaste"
    }
]

print("Bắt đầu sao chép dữ liệu...\n")

for task in datasets_to_copy:
    src = task["src"]
    dst = task["dst"]
    
    if os.path.exists(src):
        # Tạo thư mục đích và các thư mục cha nếu chưa có
        os.makedirs(dst, exist_ok=True)
        
        # Sao chép toàn bộ nội dung bên trong src vào dst
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"Thành công: Nội dung inside '{os.path.basename(src)}' -> '{dst}'")
    else:
        print(f"Thất bại: Không tìm thấy thư mục nguồn: {src}")

print("\nHoàn thành tất cả các tác vụ!")

Bắt đầu sao chép dữ liệu...

✔️ Thành công: Nội dung inside 'dataset-resized' -> '/kaggle/working/waste-detection2-Stage/data/external/TrashNet'
✔️ Thành công: Nội dung inside 'taco-trash-detection-dataset' -> '/kaggle/working/waste-detection2-Stage/data/raw'
✔️ Thành công: Nội dung inside 'RealWaste' -> '/kaggle/working/waste-detection2-Stage/data/external/RealWaste'

Hoàn thành tất cả các tác vụ!


In [ ]:
!pip install -q ultralytics timm torch torchvision

# 0. Làm sạch dữ liệu gốc (Loại bỏ các bounding box lỗi)
!python src/data_prep/data_cleaning.py

# 1. Tự động sinh cả 2 bộ YOLO (1 class và 5 class)
!python src/Training_dataYolo.py

# 2. Chia Train/Val/Test
!python src/data_prep/split_dataset.py

# 3. Cắt rác từ TACO
!python src/data_prep/crop_for_classification.py

# 4. Gộp với TrashNet và RealWaste
!python src/data_prep/merge_external_datasets.py

print("Xong tiền xử lý! Dữ liệu đã sẵn sàng.")

In [ ]:
!python src/train_stage1_detector.py
!python src/train_stage2_classifier.py